# Running OMOS on MetaWorld MT50

In [1]:
import sys
sys.path.append('../')
import jax
import jax.numpy as jnp
import numpy as np
import pickle
from jax_meta_rl.meta_rl.algorithms.omos import OMOS

# --- Load Dataset ---
with open('../data/mt50_dataset.pkl', 'rb') as f:
    dataset = pickle.load(f)

task_names = list(dataset.keys())
print(f"Loaded dataset with {len(task_names)} tasks.")

# --- Hyperparameters ---
OBS_DIM = 39 # From MetaWorld MT50
ACTION_DIM = 4 # From MetaWorld MT50
META_BATCH_SIZE = 8
SUPPORT_SET_SIZE = 1 # Number of trajectories
QUERY_SET_SIZE = 1
NUM_META_UPDATES = 100
KEY = jax.random.PRNGKey(42)

# --- Data Sampler ---
def sample_meta_batch(key, meta_batch_size):
    batch_task_names = np.random.choice(task_names, size=meta_batch_size, replace=False)
    tasks = []
    for task_name in batch_task_names:
        support_traj = dataset[task_name][0]
        query_traj = dataset[task_name][1]
        support_batch = (support_traj['observations'], support_traj['actions'], support_traj['rewards'], support_traj['next_observations'], support_traj['terminals'])
        query_batch = (query_traj['observations'], query_traj['actions'], query_traj['rewards'], query_traj['next_observations'], query_traj['terminals'])
        tasks.append((support_batch, query_batch))
    return jax.tree_util.tree_map(lambda *xs: jnp.stack(xs), *tasks)

# --- Initialize Algorithm ---
omos_agent = OMOS(obs_dim=OBS_DIM, action_dim=ACTION_DIM)
KEY, init_key = jax.random.split(KEY)
params, opt_states = omos_agent.init_params(init_key)

# --- Meta-Training Loop ---
print("Starting meta-training...")
for i in range(NUM_META_UPDATES):
    KEY, batch_key = jax.random.split(KEY)
    meta_batch = sample_meta_batch(batch_key, META_BATCH_SIZE)

    KEY, train_key = jax.random.split(KEY)
    params, opt_states['main'], total_loss = omos_agent.outer_update(params, opt_states['main'], meta_batch, train_key)

    KEY, online_key = jax.random.split(KEY)
    some_traj = dataset[task_names[0]][0]
    online_batch = (some_traj['observations'], some_traj['actions'], some_traj['next_observations'])
    params, opt_states, dynamics_loss = omos_agent.self_supervised_update(params, opt_states, online_batch)

    if (i + 1) % 10 == 0:
        print(f"Update {i+1}/{NUM_META_UPDATES} | Meta Loss: {total_loss:.4f} | Dynamics Loss: {dynamics_loss:.4f}")

print("\nMeta-training complete.")

Loaded dataset with 50 tasks.


Starting meta-training...


Update 10/100 | Meta Loss: 1.6903 | Dynamics Loss: 0.0219


Update 20/100 | Meta Loss: 20.3821 | Dynamics Loss: 0.0120


Update 30/100 | Meta Loss: -0.0167 | Dynamics Loss: 0.0071


...

Update 60/100 | Meta Loss: 19.3747 | Dynamics Loss: 0.0050


Update 70/100 | Meta Loss: -1.3218 | Dynamics Loss: 0.0048


Update 80/100 | Meta Loss: -1.4751 | Dynamics Loss: 0.0046


Update 90/100 | Meta Loss: -1.2432 | Dynamics Loss: 0.0045


Update 100/100 | Meta Loss: -0.4339 | Dynamics Loss: 0.0043

Meta-training complete.
